In [ ]:
from google.colab import drive, userdata
from huggingface_hub import login
import torch

drive.mount("/content/drive")
login(token=userdata.get("HF_TOKEN"))

BASE = "/content/drive/MyDrive/Tesi-Uninsubria"
DATA = f"{BASE}/data"
RESULTS = f"{BASE}/results"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

Mounted at /content/drive
device: cuda


In [ ]:
from transformers import AutoProcessor, AutoModel

model = AutoModel.from_pretrained("google/medsiglip-448").to(device)
processor = AutoProcessor.from_pretrained("google/medsiglip-448")

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.51GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/360 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/809 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  798kB            

spiece.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/455 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.40M [00:00<?, ?B/s]

In [ ]:
!pip install -q medmnist

In [ ]:
from medmnist import PneumoniaMNIST

train_ds = PneumoniaMNIST(split="train", download=True, size=224)

100%|██████████| 214M/214M [02:00<00:00, 1.79MB/s]


In [ ]:
import time

def embed(images, batch_size=8):
    out = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i + batch_size]
        inputs = processor(images=batch, return_tensors="pt").to(device)
        with torch.no_grad():
            feats = model.get_image_features(**inputs).pooler_output
        feats = feats / feats.norm(dim=-1, keepdim=True)
        out.append(feats.cpu())
    return torch.cat(out)

imgs = [train_ds[i][0].convert("RGB") for i in range(8)]
t0 = time.time()
E = embed(imgs)
print(E.shape)
print(E.norm(dim=-1))
print(time.time() - t0, "secondi")

torch.Size([8, 1152])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
5.200272798538208 secondi


In [ ]:
with torch.no_grad():
    chk = model(**processor(text=["a photo"], images=imgs[:1],
                            padding="max_length", return_tensors="pt").to(device))

print((chk.image_embeds[0].cpu() - E[0]).abs().max())

tensor(5.2899e-07)


In [ ]:
import os
import numpy as np

os.makedirs(f"{RESULTS}/emb", exist_ok=True)

def embed_split(ds, name, chunk=500):
    n = len(ds)
    np.save(f"{RESULTS}/emb/{name}_labels.npy", ds.labels.ravel())
    for start in range(0, n, chunk):
        path = f"{RESULTS}/emb/{name}_{start}.pt"
        if os.path.exists(path):
            continue
        imgs = [ds[i][0].convert("RGB") for i in range(start, min(start + chunk, n))]
        torch.save(embed(imgs), path)
        print(name, start, "salvato")

In [ ]:
val_ds = PneumoniaMNIST(split="val", download=True, size=224)
test_ds = PneumoniaMNIST(split="test", download=True, size=224)

t0 = time.time()
embed_split(val_ds, "val")
print((time.time() - t0) / 60, "minuti")

0.01847933530807495 minuti


In [ ]:
def load_split(name):
    labels = np.load(f"{RESULTS}/emb/{name}_labels.npy")
    parts = [torch.load(f"{RESULTS}/emb/{name}_{s}.pt")
             for s in range(0, len(labels), 500)]
    return torch.cat(parts), labels

Xv, yv = load_split("val")
print(Xv.shape, yv.shape)
print(np.bincount(yv))

torch.Size([524, 1152]) (524,)
[135 389]


In [ ]:
#dati di train ci metterà 30 min circa in teoria
t0 = time.time()
embed_split(test_ds, "test")
embed_split(train_ds, "train")
print((time.time() - t0) / 60, "minuti")

0.02130817174911499 minuti


In [ ]:
Xtr, ytr = load_split("train")
Xva, yva = load_split("val")
Xte, yte = load_split("test")
print(Xtr.shape, Xva.shape, Xte.shape)

torch.Size([4708, 1152]) torch.Size([524, 1152]) torch.Size([624, 1152])


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

clf = LogisticRegression(max_iter=1000)
clf.fit(Xtr.numpy(), ytr)

s_tr = clf.decision_function(Xtr.numpy())
s_va = clf.decision_function(Xva.numpy())
print("AUC train:", roc_auc_score(ytr, s_tr))
print("AUC val:  ", roc_auc_score(yva, s_va))

AUC train: 0.9971867517768752
AUC val:   0.9966866609540131


In [ ]:
from sklearn.metrics import confusion_matrix

pred = clf.predict(Xva.numpy())
cm = confusion_matrix(yva, pred)
tn, fp, fn, tp = cm.ravel()

print(cm)
print("recall (sensibilità):", tp / (tp + fn))
print("specificità:         ", tn / (tn + fp))
print("precisione:          ", tp / (tp + fp))
print("accuratezza:         ", (tp + tn) / len(yva))

[[128   7]
 [  6 383]]
recall (sensibilità): 0.9845758354755784
specificità:          0.9481481481481482
precisione:           0.982051282051282
accuratezza:          0.9751908396946565


In [ ]:
for t in [-3, -2, -1, 0, 1, 2]:
    pred = (s_va >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(yva, pred).ravel()
    print(f"soglia {t:>2}: recall={tp/(tp+fn):.3f}  "
          f"specificità={tn/(tn+fp):.3f}  FN={fn}  FP={fp}")

soglia -3: recall=1.000  specificità=0.267  FN=0  FP=99
soglia -2: recall=1.000  specificità=0.563  FN=0  FP=59
soglia -1: recall=0.997  specificità=0.837  FN=1  FP=22
soglia  0: recall=0.985  specificità=0.948  FN=6  FP=7
soglia  1: recall=0.941  specificità=0.993  FN=23  FP=1
soglia  2: recall=0.895  specificità=0.993  FN=41  FP=1


In [ ]:
s_te = clf.decision_function(Xte.numpy())
print("AUC test:", roc_auc_score(yte, s_te))

for t in [0, -1]:
    pred = (s_te >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
    print(f"soglia {t:>2}: recall={tp/(tp+fn):.3f}  "
          f"specificità={tn/(tn+fp):.3f}  FN={fn}  FP={fp}  "
          f"accuratezza={(tp+tn)/len(yte):.3f}")

AUC test: 0.9925158886697347
soglia  0: recall=0.992  specificità=0.863  FN=3  FP=32  accuratezza=0.944
soglia -1: recall=1.000  specificità=0.705  FN=0  FP=69  accuratezza=0.889


In [ ]:
print(clf.coef_.shape)
print(clf.intercept_.shape)
print(clf.coef_[0][:5])

(1, 1152)
(1,)
[ 0.07585102 -0.37025663  0.23748399 -0.21003613 -0.17120162]


In [ ]:
Xva, yva = load_split("val")
x = Xva[0].numpy()

mio = x @ clf.coef_[0] + clf.intercept_[0]
suo = clf.decision_function(Xva[:1].numpy())[0]
print(mio, suo)

7.24153468389909 7.24153468389909
